# Large K/V Tensor Matrix Operation Benchmark

This notebook keeps the KV-cache motivation, but the main experiment is simple: create large tensors and measure how long matrix operations take.

Why this is related to KV cache:

During decoding, attention compares the current query vector with all cached key vectors. At a simplified level, this is a matrix-vector operation:

```text
scores = K @ q
```

For long context lengths and large models, K and V tensors become very large. So before running a 100B-scale model, we can still study the core issue locally: how matrix operation latency changes as tensor size increases.


## Setup

Run this first. It picks `cuda`, `mps`, or `cpu` automatically and defines small helper functions for timing.

In [20]:
import gc
import time

import torch

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

# float16 is common for inference and lets larger tensors fit in memory.
dtype = torch.float16

print("device:", device)
print("dtype:", dtype)
print("torch:", torch.__version__)


def sync_device():
    if device.type == "cuda":
        torch.cuda.synchronize()
    elif device.type == "mps":
        torch.mps.synchronize()


def clear_memory():
    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()
    elif device.type == "mps":
        torch.mps.empty_cache()


def bytes_to_mib(n):
    return n / (1024 ** 2)


def bench(fn, repeats=10, warmup=3):
    for _ in range(warmup):
        out = fn()
    sync_device()

    t0 = time.perf_counter()
    for _ in range(repeats):
        out = fn()
    sync_device()
    t1 = time.perf_counter()

    return (t1 - t0) / repeats, out


device: mps
dtype: torch.float16
torch: 2.4.0


## KV-Cache Size Reminder

For one layer, one K tensor has this size:

```text
batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size
```

A full KV cache stores both K and V for every layer:

```text
2 * num_layers * batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size
```

This section only estimates memory. The later sections measure operation time.

In [21]:
def one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size=2):
    return batch_size * cached_tokens * num_kv_heads * head_dim * dtype_size


def full_kv_bytes(batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size=2):
    return 2 * num_layers * one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size)


def show_kv_estimate(name, batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size=2):
    one_k = one_k_bytes(batch_size, cached_tokens, num_kv_heads, head_dim, dtype_size)
    full = full_kv_bytes(batch_size, cached_tokens, num_layers, num_kv_heads, head_dim, dtype_size)
    print(name)
    print(f"  one K tensor: {bytes_to_mib(one_k):,.2f} MiB")
    print(f"  full KV cache: {full / (1024 ** 3):,.2f} GiB")

show_kv_estimate(
    "SmolLM2-135M-like, L=4096",
    batch_size=1,
    cached_tokens=4096,
    num_layers=30,
    num_kv_heads=3,
    head_dim=64,
)

show_kv_estimate(
    "Larger-model-like, L=32768",
    batch_size=1,
    cached_tokens=32768,
    num_layers=80,
    num_kv_heads=8,
    head_dim=128,
)


SmolLM2-135M-like, L=4096
  one K tensor: 1.50 MiB
  full KV cache: 0.09 GiB
Larger-model-like, L=32768
  one K tensor: 64.00 MiB
  full KV cache: 10.00 GiB


## Experiment 1: Matrix-Vector Multiply

This is the simplest version of the decode attention operation.

```text
matrix: [rows, dim]
vector: [dim]
output: [rows]
```

As `rows` increases, the vector is compared with more stored rows. In KV-cache terms, this is similar to increasing the number of cached tokens.

In [22]:
dim = 128
rows_list = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]
repeats = 10

print(f"Matrix-vector multiply: matrix [rows, {dim}] @ vector [{dim}]")
print("rows | matrix memory | avg time")
print("-" * 46)

for rows in rows_list:
    clear_memory()
    try:
        matrix = torch.randn((rows, dim), device=device, dtype=dtype)
        vector = torch.randn((dim,), device=device, dtype=dtype)

        avg_s, out = bench(lambda: matrix @ vector, repeats=repeats)
        memory_mib = bytes_to_mib(matrix.numel() * matrix.element_size())

        print(f"{rows:6d} | {memory_mib:10.2f} MiB | {avg_s * 1000:8.3f} ms")

        del matrix, vector, out
    except RuntimeError as e:
        print(f"{rows:6d} | FAILED: {str(e).splitlines()[0]}")
        break


Matrix-vector multiply: matrix [rows, 128] @ vector [128]
rows | matrix memory | avg time
----------------------------------------------
  1024 |       0.25 MiB |    0.152 ms
  2048 |       0.50 MiB |    0.131 ms
  4096 |       1.00 MiB |    0.163 ms
  8192 |       2.00 MiB |    0.267 ms
 16384 |       4.00 MiB |    0.259 ms
 32768 |       8.00 MiB |    0.383 ms
 65536 |      16.00 MiB |    1.279 ms
131072 |      32.00 MiB |    0.690 ms
262144 |      64.00 MiB |    1.877 ms
524288 |     128.00 MiB |    2.303 ms
1048576 |     256.00 MiB |    4.563 ms


## Experiment 2: Matrix-Matrix Multiply

This is heavier than matrix-vector multiply. It measures a more general dense operation:

```text
A: [n, dim]
B: [dim, n]
C: [n, n]
```

Be careful when increasing `n`, because the output matrix also becomes large.

In [23]:
dim = 256
n_list = [256, 512, 1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576]
repeats = 5

print(f"Matrix-matrix multiply: A [n, {dim}] @ B [{dim}, n]")
print("n | input memory | output memory | avg time")
print("-" * 62)

for n in n_list:
    clear_memory()
    try:
        a = torch.randn((n, dim), device=device, dtype=dtype)
        b = torch.randn((dim, n), device=device, dtype=dtype)

        avg_s, c = bench(lambda: a @ b, repeats=repeats)

        input_mib = bytes_to_mib((a.numel() + b.numel()) * a.element_size())
        output_mib = bytes_to_mib(c.numel() * c.element_size())

        print(f"{n:4d} | {input_mib:10.2f} MiB | {output_mib:11.2f} MiB | {avg_s * 1000:8.3f} ms")

        del a, b, c
    except RuntimeError as e:
        print(f"{n:4d} | FAILED: {str(e).splitlines()[0]}")
        break


Matrix-matrix multiply: A [n, 256] @ B [256, n]
n | input memory | output memory | avg time
--------------------------------------------------------------
 256 |       0.25 MiB |        0.12 MiB |    0.108 ms
 512 |       0.50 MiB |        0.50 MiB |    0.156 ms
1024 |       1.00 MiB |        2.00 MiB |    0.362 ms
2048 |       2.00 MiB |        8.00 MiB |    1.041 ms
4096 |       4.00 MiB |       32.00 MiB |    3.536 ms
8192 |       8.00 MiB |      128.00 MiB |   13.569 ms
16384 |      16.00 MiB |      512.00 MiB |   54.824 ms
32768 |      32.00 MiB |     2048.00 MiB | 4847.744 ms
65536 | FAILED: Invalid buffer size: 8.00 GB


## Experiment 3: K-Shaped Tensor Dot Product

This keeps the tensor closer to attention/KV-cache shape.

```text
K: [batch, kv_heads, cached_tokens, head_dim]
q: [batch, kv_heads, head_dim]
scores: [batch, kv_heads, cached_tokens]
```

This is still not a full transformer. It only times the K-side dot product that becomes expensive when the cache is large.

In [32]:
B = 1
H_kv = 8
D = 128
lengths = [1024, 2048, 4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576, 2097152]
repeats = 10

print("K-shaped dot product: K [B, H_kv, L, D], q [B, H_kv, D] -> scores [B, H_kv, L]")
print("L | K memory | avg time")
print("-" * 42)

for L in lengths:
    clear_memory()
    try:
        K = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)

        avg_s, scores = bench(lambda: torch.einsum("bhld,bhd->bhl", K, q), repeats=repeats)
        memory_mib = bytes_to_mib(K.numel() * K.element_size())

        print(f"{L:6d} | {memory_mib:8.2f} MiB | {avg_s * 1000:8.3f} ms")

        del K, q, scores
    except RuntimeError as e:
        print(f"{L:6d} | FAILED: {str(e).splitlines()[0]}")
        break


K-shaped dot product: K [B, H_kv, L, D], q [B, H_kv, D] -> scores [B, H_kv, L]
L | K memory | avg time
------------------------------------------
  1024 |     2.00 MiB |    0.237 ms
  2048 |     4.00 MiB |    0.278 ms
  4096 |     8.00 MiB |    0.396 ms
  8192 |    16.00 MiB |    0.714 ms
 16384 |    32.00 MiB |    0.933 ms
 32768 |    64.00 MiB |    1.109 ms
 65536 |   128.00 MiB |    2.602 ms
131072 |   256.00 MiB |    3.874 ms
262144 |   512.00 MiB |    7.186 ms
524288 |  1024.00 MiB |   13.550 ms
1048576 |  2048.00 MiB |   27.413 ms
2097152 | FAILED: Invalid buffer size: 4.00 GB


## Experiment 4: Layout Effect For K-Shaped Tensor

This compares the same K data in two layouts:

```text
BHLD: [batch, kv_heads, cached_tokens, head_dim]
BLHD: [batch, cached_tokens, kv_heads, head_dim]
```

Both store the same number of values. The timing can differ because memory strides and backend kernels access the data differently.

In [33]:
B = 1
H_kv = 8
D = 128
lengths = [4096, 8192, 16384, 32768, 65536, 131072, 262144, 524288, 1048576, 2097152]
repeats = 10

print("Layout comparison for K-shaped dot product")
print("L | BHLD time | BLHD view time | BLHD contiguous time")
print("-" * 68)

for L in lengths:
    clear_memory()
    try:
        K_bhld = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        K_blhd_view = K_bhld.transpose(1, 2)
        K_blhd = K_blhd_view.contiguous()
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)

        bhld_t, _ = bench(lambda: torch.einsum("bhld,bhd->bhl", K_bhld, q), repeats=repeats)
        blhd_view_t, _ = bench(lambda: torch.einsum("blhd,bhd->bhl", K_blhd_view, q), repeats=repeats)
        blhd_t, _ = bench(lambda: torch.einsum("blhd,bhd->bhl", K_blhd, q), repeats=repeats)

        print(f"{L:6d} | {bhld_t*1000:9.3f} ms | {blhd_view_t*1000:14.3f} ms | {blhd_t*1000:18.3f} ms")

        del K_bhld, K_blhd_view, K_blhd, q
    except RuntimeError as e:
        print(f"{L:6d} | FAILED: {str(e).splitlines()[0]}")
        break


Layout comparison for K-shaped dot product
L | BHLD time | BLHD view time | BLHD contiguous time
--------------------------------------------------------------------
  4096 |     0.541 ms |          0.411 ms |              1.225 ms
  8192 |     0.549 ms |          0.443 ms |              3.572 ms
 16384 |     0.641 ms |          0.580 ms |              3.494 ms
 32768 |     1.139 ms |          0.974 ms |              4.942 ms
 65536 |     1.718 ms |          1.658 ms |              8.460 ms
131072 |     3.200 ms |          3.203 ms |             18.612 ms
262144 |     6.409 ms |          6.522 ms |             33.975 ms
524288 |    13.916 ms |         13.252 ms |            136.040 ms
1048576 |   181.438 ms |         26.785 ms |           5072.547 ms
2097152 | FAILED: Invalid buffer size: 4.00 GB


## Diagnostic 1: Arithmetic Intensity

Arithmetic intensity tells us whether an operation is likely memory-bound or compute-bound.

```text
arithmetic_intensity = FLOPs / bytes_moved
```

For matrix-vector multiply, each matrix value is usually read once and used for only one multiply-add. That gives low arithmetic intensity, so latency is often limited by memory bandwidth.

For matrix-matrix multiply, each value can be reused many times. That gives higher arithmetic intensity, so it can become compute-bound until memory capacity becomes the bottleneck.

In [27]:
def matvec_stats(rows, dim, dtype_size=2):
    flops = 2 * rows * dim
    bytes_moved = dtype_size * (rows * dim + dim + rows)
    return flops, bytes_moved, flops / bytes_moved


def matmul_stats(m, k, n, dtype_size=2):
    flops = 2 * m * k * n
    bytes_moved = dtype_size * (m * k + k * n + m * n)
    return flops, bytes_moved, flops / bytes_moved

print("Matrix-vector examples")
for rows, dim in [(1_000_000, 64), (1_000_000, 128), (65_536, 1024)]:
    flops, bytes_moved, intensity = matvec_stats(rows, dim)
    print(f"rows={rows:9d}, dim={dim:4d} | intensity={intensity:6.2f} FLOP/byte | data={bytes_to_mib(bytes_moved):8.2f} MiB")

print("Matrix-matrix examples")
for m, k, n in [(4096, 256, 4096), (8192, 1024, 8192), (4096, 4096, 4096)]:
    flops, bytes_moved, intensity = matmul_stats(m, k, n)
    print(f"[{m}, {k}] @ [{k}, {n}] | intensity={intensity:8.2f} FLOP/byte | data={bytes_to_mib(bytes_moved):8.2f} MiB")


Matrix-vector examples
rows=  1000000, dim=  64 | intensity=  0.98 FLOP/byte | data=  123.98 MiB
rows=  1000000, dim= 128 | intensity=  0.99 FLOP/byte | data=  246.05 MiB
rows=    65536, dim=1024 | intensity=  1.00 FLOP/byte | data=  128.13 MiB
Matrix-matrix examples
[4096, 256] @ [256, 4096] | intensity=  227.56 FLOP/byte | data=   36.00 MiB
[8192, 1024] @ [1024, 8192] | intensity=  819.20 FLOP/byte | data=  160.00 MiB
[4096, 4096] @ [4096, 4096] | intensity= 1365.33 FLOP/byte | data=   96.00 MiB


## Diagnostic 2: Tall-Skinny Versus Square-ish Matrices

This separates memory-bandwidth stress from compute stress.

- Tall-skinny matrix-vector multiply reads many values but performs little reuse.
- Square-ish matrix-matrix multiply reuses values more heavily and can stress compute more.

In [28]:
cases = [
    ("tall-skinny matvec", "matvec", 1_000_000, 64, None),
    ("fatter matvec", "matvec", 262_144, 512, None),
    ("small square matmul", "matmul", 4096, 4096, 4096),
    ("medium square-ish matmul", "matmul", 8192, 1024, 8192),
]

print("case | memory | avg time")
print("-" * 58)

for label, kind, a, b, c in cases:
    clear_memory()
    try:
        if kind == "matvec":
            rows, dim = a, b
            matrix = torch.randn((rows, dim), device=device, dtype=dtype)
            vector = torch.randn((dim,), device=device, dtype=dtype)
            avg_s, out = bench(lambda: matrix @ vector, repeats=5, warmup=2)
            mem = bytes_to_mib((matrix.numel() + vector.numel() + out.numel()) * matrix.element_size())
            print(f"{label:24s} | {mem:8.2f} MiB | {avg_s*1000:9.3f} ms")
            del matrix, vector, out
        else:
            m, k, n = a, b, c
            left = torch.randn((m, k), device=device, dtype=dtype)
            right = torch.randn((k, n), device=device, dtype=dtype)
            avg_s, out = bench(lambda: left @ right, repeats=3, warmup=1)
            mem = bytes_to_mib((left.numel() + right.numel() + out.numel()) * left.element_size())
            print(f"{label:24s} | {mem:8.2f} MiB | {avg_s*1000:9.3f} ms")
            del left, right, out
    except RuntimeError as e:
        print(f"{label:24s} | FAILED: {str(e).splitlines()[0]}")


case | memory | avg time
----------------------------------------------------------
tall-skinny matvec       |   123.98 MiB |     3.002 ms
fatter matvec            |   256.50 MiB |     5.237 ms
small square matmul      |    96.00 MiB |    57.577 ms
medium square-ish matmul |   160.00 MiB |    55.811 ms


## Diagnostic 3: Single-Token Decode Versus Multi-Token Query

Single-token decode uses one query token at a time, so it tends to be memory-bound.

Prefill or chunked decoding uses many query tokens at once. That turns the operation into a larger matrix-matrix style computation and can improve hardware utilization.

Here `Q_len=1` approximates one-token decode. Larger `Q_len` approximates processing multiple query positions together.

In [29]:
B = 1
H_kv = 8
L = 16384
D = 128
q_lengths = [1, 4, 16, 64, 128]
repeats = 5

clear_memory()
K = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)

print("Q_len | output shape | avg time | ms/query-token")
print("-" * 62)

for Q_len in q_lengths:
    try:
        Q = torch.randn((B, H_kv, Q_len, D), device=device, dtype=dtype)
        avg_s, scores = bench(lambda: torch.einsum("bhqd,bhld->bhql", Q, K), repeats=repeats, warmup=2)
        print(f"{Q_len:5d} | {tuple(scores.shape)!s:18s} | {avg_s*1000:8.3f} ms | {(avg_s*1000)/Q_len:13.3f}")
        del Q, scores
    except RuntimeError as e:
        print(f"{Q_len:5d} | FAILED: {str(e).splitlines()[0]}")
        break

del K
clear_memory()


Q_len | output shape | avg time | ms/query-token
--------------------------------------------------------------
    1 | (1, 8, 1, 16384)   |    3.532 ms |         3.532
    4 | (1, 8, 4, 16384)   |    4.354 ms |         1.088
   16 | (1, 8, 16, 16384)  |    4.474 ms |         0.280
   64 | (1, 8, 64, 16384)  |    4.489 ms |         0.070
  128 | (1, 8, 128, 16384) |    5.199 ms |         0.041


## Diagnostic 4: Sweep Batch Size For K-Shaped Dot Product

Batching increases total work and memory. The interesting question is whether time increases linearly, or whether throughput improves because the backend has more parallel work to do.

In [30]:
H_kv = 8
L = 8192
D = 128
batch_sizes = [1, 2, 4, 8, 16, 32]
repeats = 5

print("batch | K memory | avg time | ms/batch-item")
print("-" * 56)

for B in batch_sizes:
    clear_memory()
    try:
        K = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)
        avg_s, scores = bench(lambda: torch.einsum("bhld,bhd->bhl", K, q), repeats=repeats, warmup=2)
        mem = bytes_to_mib(K.numel() * K.element_size())
        print(f"{B:5d} | {mem:8.2f} MiB | {avg_s*1000:8.3f} ms | {(avg_s*1000)/B:13.3f}")
        del K, q, scores
    except RuntimeError as e:
        print(f"{B:5d} | FAILED: {str(e).splitlines()[0]}")
        break


batch | K memory | avg time | ms/batch-item
--------------------------------------------------------
    1 |    16.00 MiB |    1.300 ms |         1.300
    2 |    32.00 MiB |    0.716 ms |         0.358
    4 |    64.00 MiB |    1.490 ms |         0.373
    8 |   128.00 MiB |    2.801 ms |         0.350
   16 |   256.00 MiB |    4.726 ms |         0.295
   32 |   512.00 MiB |    6.732 ms |         0.210


## Diagnostic 5: `einsum` Versus `bmm`

If a result looks strange, compare different ways of expressing the same operation. Sometimes the backend dispatch path matters as much as the mathematical operation.

Both lines below compute the same score tensor:

```python
torch.einsum("bhld,bhd->bhl", K, q)
torch.bmm(K.reshape(B * H, L, D), q.reshape(B * H, D, 1)).squeeze(-1).reshape(B, H, L)
```

In [31]:
B = 1
H_kv = 8
D = 128
lengths = [4096, 16384, 65536, 262144]
repeats = 5

print("L | einsum time | bmm time | outputs close")
print("-" * 58)

for L in lengths:
    clear_memory()
    try:
        K = torch.randn((B, H_kv, L, D), device=device, dtype=dtype)
        q = torch.randn((B, H_kv, D), device=device, dtype=dtype)

        def via_einsum():
            return torch.einsum("bhld,bhd->bhl", K, q)

        def via_bmm():
            K_flat = K.reshape(B * H_kv, L, D)
            q_flat = q.reshape(B * H_kv, D, 1)
            return torch.bmm(K_flat, q_flat).squeeze(-1).reshape(B, H_kv, L)

        einsum_t, out_e = bench(via_einsum, repeats=repeats, warmup=2)
        bmm_t, out_b = bench(via_bmm, repeats=repeats, warmup=2)
        same = torch.allclose(out_e, out_b, atol=1e-2, rtol=1e-2)

        print(f"{L:7d} | {einsum_t*1000:11.3f} ms | {bmm_t*1000:8.3f} ms | {same}")
        del K, q, out_e, out_b
    except RuntimeError as e:
        print(f"{L:7d} | FAILED: {str(e).splitlines()[0]}")
        break


L | einsum time | bmm time | outputs close
----------------------------------------------------------
   4096 |       0.515 ms |    0.442 ms | True
  16384 |       1.361 ms |    0.712 ms | True
  65536 |       2.310 ms |    1.726 ms | True
 262144 |       7.877 ms |    7.736 ms | True


## What To Report

After running the notebook, report the observed trend simply:

```text
I tested large matrix operations locally without loading a large model. I measured matrix-vector multiply, matrix-matrix multiply, and a K-shaped attention-style dot product while increasing tensor size. The time generally increases as the matrix/K tensor grows, and the machine eventually becomes memory-limited. This gives a local approximation of why long-context KV caches become expensive before moving to larger models on Colab.
```

Fill in your largest successful tensor size and the timing numbers from the tables.